# Predilex NLP — Entraînement sur Kaggle

**Avant de commencer :**
1. `Settings` (à droite) → `Accelerator` → **GPU T4 x2**
2. `Settings` → `Internet` → **On**

---
## Étape 1 — Vérification du GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU détecté : {gpu_name} ({gpu_mem:.1f} GB VRAM)")
else:
    print("❌ Pas de GPU — active le GPU : Settings > Accelerator > GPU T4")
    raise SystemExit("GPU requis")

---
## Étape 2 — Détecter les chemins des datasets

In [ ]:
import os
from pathlib import Path
import glob

# Afficher tous les fichiers disponibles dans /kaggle/input/
print("Fichiers disponibles dans /kaggle/input/ :")
for p in sorted(Path("/kaggle/input").rglob("*")):
    if p.is_file():
        print(f"  {p}")
    elif p.is_dir() and p.parent == Path("/kaggle/input"):
        print(f"  {p}/")

In [ ]:
# Auto-détection des fichiers
def find_file(pattern):
    """Cherche un fichier dans /kaggle/input/ par pattern glob."""
    results = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    return results[0] if results else None

def find_dir(name):
    """Cherche un dossier dans /kaggle/input/ par nom."""
    for p in Path("/kaggle/input").rglob("*"):
        if p.is_dir() and p.name == name:
            return str(p)
    return None

# Trouver chaque fichier
Y_TRAIN_PATH = find_file("Y_train_predilex*.csv")  # capture "Y_train_predilex (1).csv"
X_IDS_PATH   = find_file("x_train_ids.csv")
TXT_DIR      = find_dir("txt_files")

print(f"Y_train_predilex : {Y_TRAIN_PATH}")
print(f"x_train_ids      : {X_IDS_PATH}")
print(f"txt_files/       : {TXT_DIR}")

if not all([Y_TRAIN_PATH, X_IDS_PATH, TXT_DIR]):
    raise FileNotFoundError("Un ou plusieurs fichiers introuvables — vérifie l'étape 2 ci-dessus")

print("\n✅ Tous les fichiers trouvés")

---
## Étape 3 — Installation des dépendances

In [ ]:
!pip install -q transformers[torch] accelerate sentencepiece datasets pyyaml scikit-learn
print("✅ Dépendances installées")

import transformers, torch, accelerate
print(f"  torch        : {torch.__version__}")
print(f"  transformers : {transformers.__version__}")
print(f"  accelerate   : {accelerate.__version__}")

---
## Étape 4 — Cloner le repo GitHub

In [ ]:
import os
from getpass import getpass

TOKEN = getpass("GitHub token (laisser vide si repo public) : ")

if TOKEN:
    GITHUB_URL = f"https://{TOKEN}@github.com/djema-rayane/predilex-nlp.git"
else:
    GITHUB_URL = "https://github.com/djema-rayane/predilex-nlp.git"

REPO_DIR = "/kaggle/working/predilex-nlp"

if os.path.exists(REPO_DIR):
    print("Repo déjà cloné — mise à jour...")
    os.chdir(REPO_DIR)
    !git pull
else:
    !git clone {GITHUB_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f"✅ Répertoire : {os.getcwd()}")
!ls

---
## Étape 5 — Lier les données

In [ ]:
import os, shutil
from pathlib import Path

REPO_DIR = "/kaggle/working/predilex-nlp"
RAW = f"{REPO_DIR}/data/raw"

# Créer les dossiers
Path(f"{REPO_DIR}/data/raw").mkdir(parents=True, exist_ok=True)
Path(f"{REPO_DIR}/data/processed").mkdir(parents=True, exist_ok=True)
Path(f"{REPO_DIR}/models").mkdir(parents=True, exist_ok=True)

# Y_train_predilex.csv — copie avec le bon nom (le fichier Kaggle s'appelle "Y_train_predilex (1).csv")
dst_y = f"{RAW}/Y_train_predilex.csv"
if not os.path.exists(dst_y):
    shutil.copy(Y_TRAIN_PATH, dst_y)
print(f"✅ Y_train_predilex.csv")

# x_train_ids.csv
dst_x = f"{RAW}/x_train_ids.csv"
if not os.path.exists(dst_x):
    os.symlink(X_IDS_PATH, dst_x)
print(f"✅ x_train_ids.csv")

# txt_files/ — lien symbolique vers le dossier
dst_txt = f"{RAW}/txt_files"
if not os.path.exists(dst_txt):
    os.symlink(TXT_DIR, dst_txt)
n_txt = len(list(Path(dst_txt).glob("*.txt")))
print(f"✅ txt_files/ ({n_txt} fichiers .txt)")

---
## Étape 6 — Vérification du chargement

In [ ]:
import sys
REPO_DIR = "/kaggle/working/predilex-nlp"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from src.data_loader import load_config, load_train_data

cfg = load_config('configs/config.yaml')
df  = load_train_data(cfg)

print(f"\n✅ {len(df)} documents chargés")
print(f"Sexe : {df['sexe'].value_counts().to_dict()}")
print(f"Text moy : {df['text'].str.len().mean():.0f} chars")

---
## Étape 7 — Entraînement modèle SEXE (~15 min)

In [ ]:
!python src/train_sex.py --config configs/config.yaml

In [ ]:
from pathlib import Path
sex_model_path = Path('models/sex_classifier/best_model')
if sex_model_path.exists():
    files = list(sex_model_path.iterdir())
    print(f"✅ Modèle sexe sauvegardé : {len(files)} fichiers")
else:
    print("❌ Modèle introuvable — vérifier les logs ci-dessus")

---
## Étape 8 — Entraînement modèle DATES (~45 min)

In [ ]:
!python src/train_dates.py --config configs/config.yaml

In [ ]:
date_model_path = Path('models/date_classifier/best_model')
if date_model_path.exists():
    files = list(date_model_path.iterdir())
    print(f"✅ Modèle dates sauvegardé : {len(files)} fichiers")
else:
    print("❌ Modèle introuvable — vérifier les logs ci-dessus")

---
## Étape 9 — Évaluer sans réentraîner (si erreur en fin de run)

In [ ]:
!python src/eval_saved_model.py --config configs/config.yaml

---
## Étape 10 — Analyse des erreurs

In [ ]:
!python src/analyze_errors.py --config configs/config.yaml

# Pour voir plus d'exemples consolidation :
# !python src/analyze_errors.py --config configs/config.yaml --task consolidation --n 5

---
## Étape 11 — Sauvegarder les modèles

> Clique sur **Save Version** en haut à droite pour persister les outputs Kaggle.
> Ou télécharge le zip ci-dessous depuis le panneau fichiers (icône dossier à droite).

In [ ]:
import shutil, zipfile
from pathlib import Path

OUTPUT_DIR = "/kaggle/working/predilex_models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for model_name in ["sex_classifier", "date_classifier"]:
    src = f"/kaggle/working/predilex-nlp/models/{model_name}/best_model"
    dst = f"{OUTPUT_DIR}/{model_name}"
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        size_mb = sum(f.stat().st_size for f in Path(dst).rglob('*') if f.is_file()) / 1e6
        print(f"✅ {model_name} ({size_mb:.0f} MB)")
    else:
        print(f"❌ {model_name} introuvable")

# Zip pour téléchargement
zip_path = "/kaggle/working/predilex_models.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for model_name in ["sex_classifier", "date_classifier"]:
        src = Path(f"{OUTPUT_DIR}/{model_name}")
        if src.exists():
            for f in src.rglob("*"):
                if f.is_file():
                    zf.write(f, f.relative_to(OUTPUT_DIR))

size_mb = Path(zip_path).stat().st_size / 1e6
print(f"\n✅ Archive : {zip_path} ({size_mb:.0f} MB)")
print("Télécharge depuis le panneau fichiers à droite.")

---
## Étape 12 — Inférence finale (quand test set disponible)

In [ ]:
# À décommenter quand le test set Predilex est disponible
# TEST_DATA_DIR = "/kaggle/input/predilex-test"
# os.symlink(f"{TEST_DATA_DIR}/x_test_ids.csv", "data/raw/x_test_ids.csv")
# os.symlink(f"{TEST_DATA_DIR}/txt_files", "data/raw/x_test_txt_files")
# !python src/predict.py

print("Décommenter les lignes ci-dessus quand le test set est disponible.")